In [55]:
import pandas as pd
import requests
import json
from bs4 import BeautifulSoup
import time
from urllib.robotparser import RobotFileParser
import argparse
from tqdm import tqdm
import os
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, Date, func, case, select
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import relationship, sessionmaker
from datetime import date


2.1.1 Data Format Converter

1. Build a Python program that converts data between CSV, JSON, Excel, and Text
formats.

In [3]:
class DataConverter:
    
    def __init__(self, input_path):
        self.input_path = input_path
        self.df = None
        
    def read_file(self):
        ext = self.input_path.split('.')[-1].lower()
        
        if ext == 'csv':
            self.df = pd.read_csv(self.input_path)
            
        elif ext == 'json':
            with open(self.input_path) as f:
                data = json.load(f)
                
            # Handle nested JSON
            if isinstance(data, list):
                self.df = pd.json_normalize(data)
            else:
                self.df = pd.json_normalize([data])
                
        elif ext in ['xlsx', 'xls']:
            self.df = pd.read_excel(self.input_path)
            
        elif ext == 'txt':
            self.df = pd.read_csv(self.input_path, delimiter='\t')
            
        else:
            raise ValueError("Unsupported file format.")
            
        print("File loaded successfully!")
        return self.df
    
    
    def convert(self, output_path):
        ext = output_path.split('.')[-1].lower()
        
        if ext == 'csv':
            self.df.to_csv(output_path, index=False)
            
        elif ext == 'json':
            self.df.to_json(output_path, orient='records', indent=4)
            
        elif ext in ['xlsx', 'xls']:
            self.df.to_excel(output_path, index=False)
            
        elif ext == 'txt':
            self.df.to_csv(output_path, sep='\t', index=False)
            
        else:
            raise ValueError("Unsupported output format.")
            
        print(f"File converted and saved as {output_path}")

2. How will your program handle nested JSON structures during conversion?

When converting nested JSON:
->We use pandas.json_normalize()
->It flattens nested structures
->Nested keys become column names using dot notation
Example:
{
  "name": "John",
  "address": {
    "city": "New York",
    "zip": 10001
  }
}
becomes:
| name | address.city | address.zip |
| ---- | ------------ | ----------- |
|john  |new york      |10001        |

In [4]:
nested_json = {
    "name": "Alice",
    "contact": {
        "email": "alice@gmail.com",
        "phone": "123456"
    }
}

df_nested = pd.json_normalize(nested_json)
df_nested

,name,contact.email,contact.phone
0,Alice,alice@gmail.com,123456


3. How do you validate data types and detect missing values during conversion?

In [5]:
def validate_data(df):
    
    print("----- DATA TYPES -----")
    print(df.dtypes)
    
    print("\n----- MISSING VALUES -----")
    print(df.isnull().sum())
    
    print("\n----- DUPLICATE ROWS -----")
    print(df.duplicated().sum())
    
    print("\n----- SUMMARY STATISTICS -----")
    print(df.describe(include='all'))

program validates data by:
1. Checking column data types using:
df.dtypes
2. Detecting missing values using:
df.isnull().sum()
3. Detecting duplicates:
df.duplicated().sum()
4. Identifying inconsistencies via:
    ->Mixed data types
    ->Unexpected null values
    ->Outliers in numerical columns

5. Generate a data quality report showing missing values, data types, and inconsistencies.

In [6]:
def generate_quality_report(df):
    
    report = pd.DataFrame()
    
    report['Data Type'] = df.dtypes
    report['Missing Values'] = df.isnull().sum()
    report['Missing %'] = (df.isnull().sum() / len(df)) * 100
    report['Unique Values'] = df.nunique()
    
    report['Sample Value'] = df.iloc[0]
    
    return report

4. Design a command-line interface (CLI) for selecting input and output formats.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Data Format Converter")
    
    parser.add_argument("input", help="Input file path")
    parser.add_argument("output", help="Output file path")
    
    args = parser.parse_args()
    
    converter = DataConverter(args.input)
    df = converter.read_file()
    
    validate_data(df)
    
    report = generate_quality_report(df)
    print("\nData Quality Report:\n", report)
    
    converter.convert(args.output)
main()

2.2.1 Student Management System

6. Design a Relational Database Schema

In [10]:
Base = declarative_base()

# Students table
class Student(Base):
    __tablename__ = 'students'
    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    email = Column(String, unique=True, nullable=False)
    enrollments = relationship("Enrollment", back_populates="student")
    attendance_records = relationship("Attendance", back_populates="student")

# Courses table
class Course(Base):
    __tablename__ = 'courses'
    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    credits = Column(Integer, nullable=False)
    enrollments = relationship("Enrollment", back_populates="course")
    attendance_records = relationship("Attendance", back_populates="course")

# Enrollments table
class Enrollment(Base):
    __tablename__ = 'enrollments'
    id = Column(Integer, primary_key=True)
    student_id = Column(Integer, ForeignKey('students.id'))
    course_id = Column(Integer, ForeignKey('courses.id'))
    grade = Column(Float)  # Assuming 0.0 - 4.0 scale
    student = relationship("Student", back_populates="enrollments")
    course = relationship("Course", back_populates="enrollments")

# Attendance table
class Attendance(Base):
    __tablename__ = 'attendance'
    id = Column(Integer, primary_key=True)
    student_id = Column(Integer, ForeignKey('students.id'))
    course_id = Column(Integer, ForeignKey('courses.id'))
    date = Column(Date)
    status = Column(String)  # e.g., "Present", "Absent"
    student = relationship("Student", back_populates="attendance_records")
    course = relationship("Course", back_populates="attendance_records")

# Create the SQLite database
engine = create_engine('sqlite:///school.db')
Base.metadata.create_all(engine)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

C:\Users\shahi\AppData\Local\Temp\ipykernel_6728\2483821873.py:1: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [19]:
# --- Add students ---
students = [
    Student(name="Alice Johnson", email="alice@example.com"),
    Student(name="Bob Smith", email="bob@example.com"),
    Student(name="Charlie Lee", email="charlie@example.com")
]

session.add_all(students)
session.commit()

# --- Add courses ---
courses = [
    Course(name="Mathematics", credits=3),
    Course(name="Physics", credits=4),
    Course(name="History", credits=2)
]

session.add_all(courses)
session.commit()

# --- Add enrollments with grades ---
enrollments = [
    Enrollment(student_id=1, course_id=1, grade=3.5),
    Enrollment(student_id=1, course_id=2, grade=3.0),
    Enrollment(student_id=1, course_id=3, grade=4.0),
    
    Enrollment(student_id=2, course_id=1, grade=2.0),
    Enrollment(student_id=2, course_id=2, grade=2.5),
    Enrollment(student_id=2, course_id=3, grade=1.5),
    
    Enrollment(student_id=3, course_id=1, grade=4.0),
    Enrollment(student_id=3, course_id=2, grade=3.8),
    Enrollment(student_id=3, course_id=3, grade=3.9),
]

session.add_all(enrollments)
session.commit()

# --- Add attendance records ---
attendance_records = [
    # Alice
    Attendance(student_id=1, course_id=1, date=date(2026,2,1), status="Present"),
    Attendance(student_id=1, course_id=1, date=date(2026,2,2), status="Absent"),
    Attendance(student_id=1, course_id=2, date=date(2026,2,1), status="Present"),
    Attendance(student_id=1, course_id=3, date=date(2026,2,1), status="Present"),
    
    # Bob
    Attendance(student_id=2, course_id=1, date=date(2026,2,1), status="Absent"),
    Attendance(student_id=2, course_id=2, date=date(2026,2,1), status="Present"),
    Attendance(student_id=2, course_id=3, date=date(2026,2,1), status="Absent"),
    
    # Charlie
    Attendance(student_id=3, course_id=1, date=date(2026,2,1), status="Present"),
    Attendance(student_id=3, course_id=2, date=date(2026,2,1), status="Present"),
    Attendance(student_id=3, course_id=3, date=date(2026,2,1), status="Present"),
]

session.add_all(attendance_records)
session.commit()

7. Write SQL queries to calculate the GPA of each student.

In [22]:
gpa_query = session.query(
    Student.id,
    Student.name,
    (func.sum(Enrollment.grade * Course.credits) / func.sum(Course.credits)).label('GPA')
).select_from(Student).join(Enrollment, Student.id == Enrollment.student_id)\
  .join(Course, Enrollment.course_id == Course.id)\
  .group_by(Student.id)

for student_id, student_name, gpa in gpa_query:
    print(f"{student_name} (ID: {student_id}) - GPA: {gpa:.2f}")

Alice Johnson (ID: 1) - GPA: 3.39
Bob Smith (ID: 2) - GPA: 2.11
Charlie Lee (ID: 3) - GPA: 3.89


8. Generate attendance reports for individual students and courses.

In [23]:
#a) Attendance report for a specific student
student_id = 1  # Example student

attendance_report = session.query(
    Course.name,
    Attendance.date,
    Attendance.status
).join(Course).filter(Attendance.student_id == student_id).order_by(Course.name, Attendance.date)

for course_name, date, status in attendance_report:
    print(f"{course_name} - {date} - {status}")

#b) Attendance report for a specific course
course_id = 1  # Example course

attendance_report = session.query(
    Student.name,
    Attendance.date,
    Attendance.status
).join(Student).filter(Attendance.course_id == course_id).order_by(Student.name, Attendance.date)

for student_name, date, status in attendance_report:
    print(f"{student_name} - {date} - {status}")

History - 2026-02-01 - Present
Mathematics - 2026-02-01 - Present
Mathematics - 2026-02-02 - Absent
Physics - 2026-02-01 - Present
Alice Johnson - 2026-02-01 - Present
Alice Johnson - 2026-02-02 - Absent
Bob Smith - 2026-02-01 - Absent
Charlie Lee - 2026-02-01 - Present


9. Analyze course performance using enrollment and grade data.

In [24]:
course_performance = session.query(
    Course.name,
    func.avg(Enrollment.grade).label("average_grade")
).join(Enrollment).group_by(Course.id)

for course_name, avg_grade in course_performance:
    print(f"{course_name} - Average Grade: {avg_grade:.2f}")

Mathematics - Average Grade: 3.17
Physics - Average Grade: 3.10
History - Average Grade: 3.13


10. Identify at-risk students based on grades and attendance patterns.

In [27]:
# --- Attendance percentage per student ---
attendance_stats = session.query(
    Student.id,
    Student.name,
    (func.sum(case((Attendance.status=='Present', 1), else_=0)) * 1.0 / func.count(Attendance.id)).label('attendance_rate')
).select_from(Student).join(Attendance, Student.id == Attendance.student_id)\
  .group_by(Student.id).all()

# --- GPA per student ---
gpa_stats = session.query(
    Student.id,
    (func.sum(Enrollment.grade * Course.credits) / func.sum(Course.credits)).label('GPA')
).select_from(Student).join(Enrollment, Student.id == Enrollment.student_id)\
  .join(Course, Enrollment.course_id == Course.id)\
  .group_by(Student.id).all()

# --- Combine GPA and attendance to identify at-risk students ---
gpa_dict = {s.id: s.GPA for s in gpa_stats}
at_risk_students = []

for s in attendance_stats:
    gpa = gpa_dict.get(s.id, 0)
    if gpa < 2.0 or s.attendance_rate < 0.75:  # GPA < 2.0 or attendance < 75%
        at_risk_students.append((s.name, gpa, s.attendance_rate))

# --- Print at-risk students ---
for name, gpa, attendance in at_risk_students:
    print(f"At-Risk: {name} - GPA: {gpa:.2f}, Attendance: {attendance*100:.1f}%")

At-Risk: Bob Smith - GPA: 2.11, Attendance: 33.3%


2.3.1Accessing and processing data from APIs (REST, SOAP)

11. Fetch weather data from at least two different weather APIs.

In [ ]:
OWM_API_KEY = "your_openweathermap_api_key"
WA_API_KEY = "your_weatherapi_key"#dont have an api key

city = "kathmandu"

# OpenWeatherMap
owm_url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OWM_API_KEY}&units=metric"
owm_response = requests.get(owm_url)
owm_data = owm_response.json()
print("OpenWeatherMap:", owm_data)

# WeatherAPI
wa_url = f"http://api.weatherapi.com/v1/current.json?key={WA_API_KEY}&q={city}&aqi=no"
wa_response = requests.get(wa_url)
wa_data = wa_response.json()
print("WeatherAPI:", wa_data)

12. How do you securely manage API keys in your application?

In [ ]:
# Store keys in environment variables
OWM_API_KEY = os.getenv("OWM_API_KEY")
WA_API_KEY = os.getenv("WA_API_KEY")
#OR
from dotenv import load_dotenv
load_dotenv()

OWM_API_KEY = os.getenv("OWM_API_KEY")
WA_API_KEY = os.getenv("WA_API_KEY")

13. Handle API rate limits and failed requests gracefully.

In [ ]:
def fetch_json(url, retries=3):
    for attempt in range(retries):
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:  # Rate limit exceeded
            print("Rate limit hit, retrying...")
            import time; time.sleep(2 * (attempt + 1))
        else:
            print(f"Request failed: {response.status_code}")
    return None

owm_data = fetch_json(owm_url)
wa_data = fetch_json(wa_url)

14. Normalize weather data obtained from different APIs into a common format.

In [ ]:
def normalize_weather(owm, wa):
    normalized = []

    # OpenWeatherMap
    if owm:
        normalized.append({
            "source": "OpenWeatherMap",
            "temp_c": owm['main']['temp'],
            "humidity": owm['main']['humidity'],
            "condition": owm['weather'][0]['description']
        })

    # WeatherAPI
    if wa:
        normalized.append({
            "source": "WeatherAPI",
            "temp_c": wa['current']['temp_c'],
            "humidity": wa['current']['humidity'],
            "condition": wa['current']['condition']['text']
        })
    return normalized

weather_normalized = normalize_weather(owm_data, wa_data)
weather_normalized

15. Compare daily weather reports and forecasts from multiple sources.

In [ ]:
for w in weather_normalized:
    print(f"{w['source']}: {w['temp_c']}°C, {w['humidity']}% humidity, {w['condition']}")

# Simple comparison
if len(weather_normalized) == 2:
    diff = abs(weather_normalized[0]['temp_c'] - weather_normalized[1]['temp_c'])
    print(f"Temperature difference between sources: {diff:.1f}°C")

16. Implement a basic alert system based on weather conditions.

In [ ]:
alerts = []
for w in weather_normalized:
    if w['temp_c'] > 30:
        alerts.append(f"{w['source']} alert: High temperature {w['temp_c']}°C")
    if 'rain' in w['condition'].lower():
        alerts.append(f"{w['source']} alert: Rain expected ({w['condition']})")

for alert in alerts:
    print(alert)

2.4.1 Web scraping using requests and BeautifulSoup

17. Scrape news articles from multiple websites while following ethical scraping practices.

In [40]:
def is_allowed(url, user_agent="*"):
    base_url = "/".join(url.split("/")[:3])
    robots_url = base_url + "/robots.txt"
    
    rp = RobotFileParser()
    rp.set_url(robots_url)
    rp.read()
    
    return rp.can_fetch(user_agent, url)
#scrape bbc headlines
bbc_url = "https://www.bbc.com/news"

if is_allowed(bbc_url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(bbc_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    headlines = []
    for h in soup.find_all("h2")[:10]:
        headlines.append(h.get_text(strip=True))

    print("BBC Headlines:")
    for h in headlines:
        print("-", h)
else:
    print("Scraping not allowed by robots.txt")

time.sleep(2)  # polite delay

BBC Headlines:
- Trump's theatrical State of the Union address offers little hint of any change in course
- Fact-checking Trump's longest ever State of the Union
- BBC sees damage in Puerto Vallarta after Mexican cartel violence
- King asked to save UK's oldest Indian restaurant
- Japan to deploy missiles on island near Taiwan by 2031
- Don't break up NewJeans and I'll forgo $18m payout, says ex-K-pop boss
- Chinese dance group's tour triggers bomb threat against Australian PM
- Modi's Israel visit to test India's priorities in the Middle East
- Threat of further violence looms after Mexican cartel rampage
- German chancellor lands in Beijing for inaugural China trip


18. How do you ensure compliance with robots.txt and terms of service?
we can ensure compliance by:
1. using RobotFileParser to check permission.
2. reading website robots.txt manually.
3. not scrapping:
    ->login proctected page
    ->paid content.
4. adding delays
5.setting a proper user-agent header.
6. not overlaoding servers.

19. Extract headlines, full content, authors, publication dates, and categories from news
pages.

In [50]:
url="https://pnews.com.np/"
def scrape_article(url):
    if not is_allowed(url):
        print("Scraping not allowed by robots.txt")
        return None
    
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    title = soup.find("h1")
    author = soup.find("span", class_="author")
    date = soup.find("time")
    category = soup.find("meta", property="article:section")
    paragraphs = soup.find_all("p")

    return {
        "title": title.get_text(strip=True) if title else None,
        "author": author.get_text(strip=True) if author else "Unknown",
        "date": date.get_text(strip=True) if date else None,
        "category": category["content"] if category else None,
        "content": " ".join([p.get_text() for p in paragraphs])
    }
article_data = scrape_article(url)
print(article_data)

{'title': None, 'author': 'पिपुल्स न्युज', 'date': 'February 24, 2026', 'category': None, 'content': '  People’s News : चुनाव आउन जम्मा नौ दिन बाँकी रहँदा मुस्ताङ … Read more   People’s News : सार्वजनिक पदको दुरुपयोग गरेको आशङ्कामा सोमबार पक्राउ परेको … Read more   People’s News Kathmandu : सैन्य शासनको प्रयास र विद्रोहको योजना बनाएको … Read more   People’s News : सर्वोच्च अदालतले सन् १९७७ मा लागू भएको ‘इन्टर्नेसनल … Read more   People’s News प्रतिबन्धित लागुऔषधvसहित दुईजनालाई पक्राउ सिराहाको सीमावर्ती भगवानपुर गाउँपालिका– ३ … Read more   People’s News kathmandu : आगामी फागुन २१ गते हुने प्रतिनिधिसभा निर्वाचन … Read more   People’s News Kathmandu : राष्ट्रिय स्वतन्त्र पार्टी (रास्वपा) का वरिष्ठ नेता … Read more   People’s News : घरघरमा गेरु रङ्गको झण्डा, केहीमा हनुमान्\u200cको तस्बिर । … Read more   People’s News : कृत्रिम बौद्धिकता (एआई) ले परम्परागत गहना उद्योगमा नयाँ … Read more   People’s News : टी-२० विश्वकपको ३६ औं खेल आज श्रीलंका र … Read more   People’s News kathmandu :\xa0राष्

20. Store the scraped news data in a structured format for analysis.

In [52]:
articles = []
articles.append(scrape_article("https://pnews.com.np/"))

df = pd.DataFrame(articles)
df.head()

,title,author,date,category,content
0,None,पिपुल्स न्युज,"February 24, 2026",None,People’s News : चुनाव आउन जम्मा नौ दिन बाँकी...


2.5.1 Handling large datasets with chunking and lazy evaluation

22. Process CSV files that are larger than available system memory.

In [ ]:
import pandas as pd

file_path = "large_dataset.csv"

chunk_size = 100_000
chunks = pd.read_csv(file_path, chunksize=chunk_size)

for i, chunk in enumerate(chunks):
    chunk["processed_column"] = chunk["some_column"] * 2
    
    chunk.to_csv("processed_large_dataset.csv", mode="a", index=False, header=(i==0))
    
    print(f"Processed chunk {i+1}")

23. Explain how chunk-based processing works in pandas.

1. Iterator-based reading: pd.read_csv(chunksize-N) returns an iterator (TextFileReader) instead of the whole DataFrame.
2. Lazy evaluation: Only the current chunk is in memory. Old chunks can be processed and discarded.
3. Streaming computation: You can aggregate stats across chunks without holding all data.
Example: calculate sum of a column in a memory-efficient way:
total = 0
for chunk in pd.read_csv(file_path, chunksize=100_000):
    total += chunk["some_column"].sum()
print("Total sum:", total)

24. Monitor and limit memory usage while processing large datasets.
->
    1. use dtypes to reduce memory footprints:
    df = pd.read_csv(file_path, dtype={"id": "int32", "category": "category"})

    2.delete temp objects to free memory:
    import gc
    for chunk in pd.read_csv(file_path, chunksize=100_000):
    # process chunk
    del chunk
    gc.collect()

    3. monitor memory with  psutil:
    import psutil
    print(f"Memory usage: {psutil.virtual_memory().percent}%")

25. Optimize file I/O operations for large-scale data.
-> 1. use binary formats like   parquet
        # Write chunked data to Parquet
        for i, chunk in enumerate(pd.read_csv(file_path, chunksize=100_000)):
        chunk.to_parquet("processed_dataset.parquet", engine="pyarrow", compression="snappy", index=False)
    2.only read neccessary columns:
        cols = ["id", "value", "category"]
        pd.read_csv(file_path, usecols=cols, chunksize=100_000)

26. Track and display progress while processing large files.

In [ ]:
chunksize = 100_000
reader = pd.read_csv(file_path, chunksize=chunksize)

for chunk in tqdm(reader, desc="Processing chunks"):
    # Process chunk
    chunk["processed_column"] = chunk["some_column"] ** 2

Capstone Project: Integrated Data Engineering Platform

28. Choose a real-world domain and identify relevant data sources.
# Domain: Real Estate Market Analysis

# Relevant data sources:
# 1. APIs: OpenWeatherMap API for weather data
# 2. CSV files: historical housing prices
# 3. Web scraping: property listings from a sample website

29. Design a complete data pipeline including ingestion, storage, processing, API, and
visualization layers.
# Layers of the pipeline:

# 1. Ingestion: APIs, CSV files, Web scraping
# 2. Storage: SQLite database using SQLAlchemy
# 3. Processing: pandas for cleaning and merging
# 4. API: FastAPI to expose processed data
# 5. Visualization: Plotly or Matplotlib

30. Integrate multiple data sources (APIs, files, or web scraping) into a single platform.

In [ ]:
api_url = "https://api.openweathermap.org/data/2.5/weather"
params = {"q": "New York", "appid": "YOUR_API_KEY"}  # dont have a key
weather_data = requests.get(api_url, params=params).json()

weather_df = pd.DataFrame([{
    "city": "New York",
    "temperature": weather_data['main']['temp'],
    "weather": weather_data['weather'][0]['description']
}])

# CSV ingestion
historical_prices = pd.read_csv("historical_prices.csv")  

# Web scraping example
from bs4 import BeautifulSoup

url = "https://example-realestate-site.com/newyork"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

listings = []
for listing in soup.select(".listing"):
    listings.append({
        "title": listing.select_one(".title").text,
        "price": float(listing.select_one(".price").text.replace("$","").replace(",","")),
        "location": listing.select_one(".location").text
    })

web_df = pd.DataFrame(listings)

# Merge all sources
merged_df = historical_prices.merge(weather_df, on="city", how="left")
merged_df = merged_df.merge(web_df, left_on="city", right_on="location", how="left")
merged_df.head()

31. Store processed data efficiently using an appropriate database or storage system.

In [ ]:
Base = declarative_base()

class Property(Base):
    __tablename__ = "properties"
    id = Column(Integer, primary_key=True)
    title = Column(String)
    city = Column(String)
    date = Column(Date)
    price = Column(Float)
    temperature = Column(Float)
    weather = Column(String)

# Create SQLite database
engine = create_engine("sqlite:///realestate.db")
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

# Insert processed data
for _, row in merged_df.iterrows():
    property_row = Property(
        title=row.get("title"),
        city=row.get("city"),
        date=row.get("date", pd.Timestamp.today()).date(),
        price=row.get("price", 0),
        temperature=row.get("temperature"),
        weather=row.get("weather")
    )
    session.add(property_row)
session.commit()

32. Expose processed data through an API layer.

In [ ]:
app = FastAPI()

@app.get("/properties")
def get_properties(city: str = None):
    session = Session(bind=engine)
    query = session.query(Property)
    if city:
        query = query.filter(Property.city == city)
    results = query.all()
    session.close()
    return [
        {
            "id": p.id,
            "title": p.title,
            "city": p.city,
            "date": str(p.date),
            "price": p.price,
            "temperature": p.temperature,
            "weather": p.weather
        } for p in results
    ]